# Astra v4 — Security Issue Detection (Fine-Tuning)

Fine-tunes a pre-trained cyber model to **detect security issues in system telemetry** using the NSL-KDD intrusion detection corpus.

🎯 **Goal**: Train a model to classify network traffic as attack/normal, detecting security issues in real-time system telemetry.

📂 **Uses your existing checkpoint**: `astra_cyber_finetune_final.npz` (or `astra_cyber_pretrain_final.npz`).

Runtime: **Settings → Runtime → GPU (T4/H100)**, Internet ON.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
import os, subprocess, shutil, json

REPO = '/content/astra'
GIT_URL = 'https://github.com/anoneurx/astra.git'

if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=False)
subprocess.run(f'git clone --depth 1 {GIT_URL} {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed'
os.chdir(REPO)
print('repo ready at', os.getcwd())

**Upload Pre-Trained Checkpoint** (or using existing one in Colab):
Since you already have `astra_cyber_finetune_final.npz`, we'll use it directly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Check if checkpoint exists in Drive
checkpoint_path = '/content/drive/MyDrive/astra_checkpoints/astra_cyber_finetune_final.npz'
if os.path.exists(checkpoint_path):
    print('✓ Checkpoint found in Drive:', checkpoint_path)
    shutil.copy(checkpoint_path, '/content/astra_cyber_finetune_final.npz')
else:
    print('✗ Checkpoint not in default Drive path')
    # List Drive astra_checkpoints
    !ls /content/drive/MyDrive/astra_checkpoints/ 2>/dev/null || echo 'Directory empty or not found'

# Fallback: check current directory
if not os.path.exists('/content/astra_cyber_finetune_final.npz'):
    from google.colab import files
    print('Upload your astra_cyber_finetune_final.npz file:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        shutil.copy(fn, '/content/astra_cyber_finetune_final.npz')
        print('Saved:', fn)

---
## STAGE 1 — Verify Tokenizer

In [ ]:
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/cyber_bpe.json'):
    subprocess.run('python tokenizer/train_tokenizer.py --config configs/tokenizer_cyber.json', shell=True, check=True)
print('Tokenizer ready')

---
## STAGE 2 — Security Detection Fine-Tuning (10k steps, ~10 min)
Fine-tunes on held-out test21 data (17 novel attack classes) to detect security issues never seen during pre-training.

In [ ]:
import os, subprocess, shutil, json
os.chdir('/content/astra')

# Create fine-tune config overriding train data to test21
cfg = json.load(open('configs/astra5m_cyber.json'))
cfg['data']['train'] = 'datasets/cyber/test21.txt'
cfg['data']['val'] = 'datasets/cyber/val.txt'
cfg['training']['max_steps'] = 10000
cfg['training']['val_every'] = 500
cfg['out_dir'] = 'checkpoints/astra5m_cyber_sec_detect'

ft_cfg_path = '/content/astra5m_cyber_sec_detect.json'
json.dump(cfg, open(ft_cfg_path, 'w'), indent=2)
print('Fine-tune config written')

# Warm-start from your checkpoint
pretrain = '/content/astra_cyber_finetune_final.npz'
assert os.path.exists(pretrain), 'Checkpoint not found!'

!mkdir -p /content/runs/security_detect

res = subprocess.run(
    f'python training/gpu_train.py --config {ft_cfg_path} --steps 10000 --out /content/runs/security_detect --resume {pretrain} --reset-step --cache-dir /content/cache',
    shell=True, capture_output=True, text=True
)
print('STDOUT:')
print(res.stdout[-3000:])
print('STDERR:')
print(res.stderr[-3000:])
if res.returncode != 0:
    raise RuntimeError('Fine-tuning failed')

SRC = '/content/runs/security_detect/resumed/final.npz'
assert os.path.exists(SRC), 'FINAL NOT PRODUCED'

dest = '/content/astra_security_final.npz'
shutil.copy(SRC, dest)
print('Security model ok:', os.path.getsize(dest), 'bytes')

from google.colab import files
files.download(dest)
print('✅ Download started for security detection model')